In [1]:
import openai
import numpy as np
import json
import time
import requests
import torch
import torch.nn as nn
from openai import OpenAI

In [2]:
# client = OpenAI(api_key="REDACTED_USE_ENV_OPENAI_API_KEY")  # Replace with actual API key

In [7]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.special import softmax
from openai import OpenAI

# ✅ Set OpenAI API Key (Modify this based on your setup)
OPENAI_API_KEY = "REDACTED_USE_ENV_OPENAI_API_KEY"  # Replace with your actual API key

# ✅ Load MIMIC-III Dataset
df = pd.read_csv("/content/drive/MyDrive/NIPS25/notebooks/PRESCRIPTIONS.csv")

# ✅ Extract Relevant Text Fields (Drug Names)
questions = df["DRUG_NAME_GENERIC"].dropna().astype(str).tolist()
num_samples = min(10, len(questions))  # Limit to 10 samples for evaluation

# ✅ Define Models
MODELS = {

    "GPT-3.5": "gpt-3.5-turbo"
}

# ✅ Bayesian Model Class (MC Dropout)
class BayesianModel(nn.Module):
    def __init__(self, base_model):
        super(BayesianModel, self).__init__()
        self.base_model = base_model
        self.dropout = nn.Dropout(p=0.3)  # Monte Carlo Dropout

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout during inference
        return logits

# ✅ Load Models & Tokenizers
loaded_models = {}

for model_name, model_path in MODELS.items():
    print(f"\nLoading {model_name}...")
    if "GPT-3.5" in model_name:
        client = OpenAI(api_key=OPENAI_API_KEY)  # Use API Key Here
        loaded_models[model_name] = client
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            device_map="cpu",  # Force CPU execution
            torch_dtype=torch.float32  # Use full precision for CPU compatibility
        )

        model.config.use_cache = False
        model.config.pretraining_tp = 1

        # Wrap model with Bayesian uncertainty mechanism
        bayesian_model = BayesianModel(model)
        loaded_models[model_name] = (model, tokenizer, bayesian_model)
        print(f"{model_name} successfully loaded!")

# ✅ Function: Generate Response with Temperature Scaling
def generate_response(model, tokenizer, input_text, temperature):
    if isinstance(model, OpenAI):
        response = model.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": input_text}],
            max_tokens=50,
            temperature=temperature
        )
        return response.choices[0].message.content
    else:
        inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        with torch.no_grad():
            output_sequences = model.generate(
                inputs["input_ids"],
                max_new_tokens=50,
                temperature=temperature,
                pad_token_id=tokenizer.eos_token_id
            )
        return tokenizer.decode(output_sequences[0], skip_special_tokens=True)

# ✅ Function: Estimate Uncertainty using MC Dropout
def estimate_uncertainty(bayesian_model, tokenizer, input_text, n_samples=10):
    inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    predictions = []

    for _ in range(n_samples):
        with torch.no_grad():
            logits = bayesian_model(inputs["input_ids"], inputs["attention_mask"])
            predictions.append(logits.cpu().numpy())

    predictions = np.array(predictions)
    mean_prediction = predictions.mean(axis=0)
    uncertainty = np.abs(predictions.std(axis=0) / (mean_prediction + 1e-6))  # Normalize uncertainty

    return mean_prediction, uncertainty

# ✅ Function: Compute Confidence Entropy
def compute_confidence_entropy(probs):
    entropy = -np.sum(probs * np.log(probs + 1e-6))  # Avoid log(0)
    return entropy

# ✅ Clinical Uncertainty Score (CUS)
def compute_cus(uncertainties):
    return np.mean(uncertainties)

# ✅ Zero-Shot Trustworthiness Index (ZTI)
def compute_zti(cus_scores, epsilon=1e-6):
    if len(cus_scores) < 2:
        return 1.0
    mean_cus = np.mean(cus_scores)
    std_cus = np.std(cus_scores)
    max_cus = max(np.max(cus_scores), epsilon)
    zti = 1 - (np.log1p(abs(mean_cus - std_cus)) / np.log1p(max_cus))
    return max(0, min(zti, 1))

# ✅ Apply Temperature Scaling & Compute Metrics
TEMPERATURES = [1.0, 0.8, 1.2]
results = []

for model_name, model_obj in loaded_models.items():
    print(f"\n====== Evaluating {model_name} on MIMIC-III ======")

    for temp in TEMPERATURES:
        print(f"\n🔹 Running at Temperature T={temp} 🔹")

        cus_scores = []
        confidence_entropies = []

        for i in range(num_samples):
            response = generate_response(*model_obj, questions[i], temperature=temp) if isinstance(model_obj, tuple) else generate_response(model_obj, None, questions[i], temperature=temp)
            print(f"Sample {i+1} - {model_name} Response (T={temp}): {response}")

            probs = np.array([len(response.split())])
            probs = probs / np.sum(probs)

            if isinstance(model_obj, tuple):
                _, uncertainty = estimate_uncertainty(model_obj[2], model_obj[1], questions[i])
                cus_score = compute_cus(probs)
                entropy = compute_confidence_entropy(probs)

                cus_scores.append(cus_score)
                confidence_entropies.append(entropy)

        zti_score = compute_zti(cus_scores)

        print(f"🔹 {model_name} (T={temp}) CUS: {np.mean(cus_scores) if cus_scores else 'N/A'}")
        print(f"🔹 {model_name} (T={temp}) ZTI: {zti_score}")
        print(f"🔹 {model_name} (T={temp}) Confidence Entropy: {np.mean(confidence_entropies) if confidence_entropies else 'N/A'}")

        results.append({
            "Model": model_name,
            "Temperature": temp,
            "CUS": np.mean(cus_scores) if cus_scores else 'N/A',
            "ZTI": zti_score,
            "Confidence Entropy": np.mean(confidence_entropies) if confidence_entropies else 'N/A'
        })

# ✅ Convert Results to DataFrame
df_results = pd.DataFrame(results)
print(df_results)
print("\n===== Evaluation Completed Successfully! =====")


<ipython-input-7-ab53b8c50f81>:14: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/NIPS25/notebooks/PRESCRIPTIONS.csv")



Loading GPT-3.5...

====== Evaluating GPT-3.5 on MIMIC-III ======

🔹 Running at Temperature T=1.0 🔹
Sample 1 - GPT-3.5 Response (T=1.0): Tacrolimus is a medication that is used to prevent organ rejection in patients who have received a kidney, liver, or heart transplant. It belongs to a class of drugs called calcineurin inhibitors, and works by suppressing the immune system to prevent
Sample 2 - GPT-3.5 Response (T=1.0): Warfarin is a medication used to prevent blood clots from forming or growing larger in the blood vessels or heart. It is commonly prescribed for conditions such as atrial fibrillation, deep vein thrombosis, pulmonary embolism, and heart
Sample 3 - GPT-3.5 Response (T=1.0): Furosemide is a medication that belongs to a class of drugs called diuretics, also known as water pills. It works by increasing the amount of urine produced by the kidneys, helping the body get rid of excess salt and water. F
Sample 4 - GPT-3.5 Response (T=1.0): Warfarin is a medication that is used